# VIA VISÃO: detecção de fissuras (Visão Computacional)

Notebook do módulo VIA VISÃO do projeto VIA CIDADÃO. Ele prepara a base SDNET2018, treina dois modelos, avalia no teste, mede a latência e processa uma sequência de imagens que simula um vídeo de monitoramento.

Antes de começar: no menu Ambiente de execução, escolha Alterar tipo de ambiente de execução e marque GPU (T4). Rode as células na ordem.

## 1. Preparar o ambiente
Monta o Drive, copia o projeto e confere a GPU. A pasta `via_cidadao_visao` (este projeto) deve estar no seu Drive em `MyDrive`, e o arquivo zip da SDNET2018 também.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print('GPU disponível:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')


In [ ]:
# Copia o projeto do Drive para o Colab e instala o que falta.
!rm -rf /content/via_cidadao_visao
!cp -r /content/drive/MyDrive/via_cidadao_visao /content/via_cidadao_visao
%cd /content/via_cidadao_visao
!pip -q install pandas scikit-learn opencv-python-headless


## 2. Base SDNET2018
Baixe o zip em https://digitalcommons.usu.edu/all_datasets/48/ (licença CC BY 4.0, cerca de 500 MB), coloque no Drive e ajuste o caminho abaixo. A célula descompacta e mostra a estrutura.

In [ ]:
ZIP_SDNET = '/content/drive/MyDrive/SDNET2018.zip'   # ajuste se o nome ou a pasta for outro
!rm -rf /content/sdnet && mkdir -p /content/sdnet
!unzip -q "$ZIP_SDNET" -d /content/sdnet
!find /content/sdnet -maxdepth 3 -type d | head -20


In [ ]:
import sys, pathlib
sys.path.insert(0, 'src')
import dados

# Acha a pasta que contém D, P e W (pode haver uma pasta a mais dentro do zip).
candidatas = [p for p in pathlib.Path('/content/sdnet').rglob('*') if p.is_dir() and {'D','P','W'} <= {q.name for q in p.iterdir() if q.is_dir()}]
RAIZ = str(candidatas[0])
print('Raiz da base:', RAIZ)

todos = dados.listar_imagens(RAIZ)
print(todos.head())
print(dados.conferir_grupos(todos))


### Conferir o grupo de cada foto
O código assume que o número antes do traço no nome do arquivo identifica a foto de origem (por exemplo, `7001-12.jpg` é o recorte 12 da foto 7001). A tabela acima mostra quantos grupos saíram por superfície. A página da base diz que há 230 fotos (54 de tabuleiros, 72 de paredes e 104 de pavimentos). Se o número de grupos for muito diferente disso, a regra do nome precisa ser revista antes de treinar, senão pode haver vazamento entre treino e teste.

## 3. Examinar algumas imagens
Uma amostra com e sem fissura em cada superfície.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
fig, eixos = plt.subplots(3, 6, figsize=(16, 8))
for i, sup in enumerate(['D', 'P', 'W']):
    for j, rot in enumerate([1, 0]):
        amostra = todos[(todos.superficie == sup) & (todos.rotulo == rot)].sample(3, random_state=1)
        for k, (_, l) in enumerate(amostra.iterrows()):
            ax = eixos[i][j * 3 + k]
            ax.imshow(Image.open(l.caminho)); ax.axis('off')
            ax.set_title(f"{sup} {'com fissura' if rot else 'sem fissura'}", fontsize=9)
plt.tight_layout(); plt.show()


## 4. Treinar, avaliar e medir latência
O script faz a divisão por foto, treina a MobileNetV3 small e a ResNet18, escolhe o limiar na validação, avalia no teste, exporta as imagens erradas e mede o tempo. Com GPU leva alguns minutos por modelo.

In [ ]:
!python src/treinar.py --raiz "$RAIZ" --saida results --epocas 4 --tamanho 224


In [ ]:
import json
met = json.load(open('results/metricas.json', encoding='utf-8'))
print('Modelo escolhido:', met['modelo_escolhido'])
for nome, r in met['modelos'].items():
    print(nome, '| limiar', r['limiar_escolhido'], '| teste', r['teste_limiar_escolhido'])


In [ ]:
# Latência do modelo escolhido na CPU, para comparar com a GPU
from video import carregar_checkpoint
from avaliacao import medir_latencia
cpu = torch.device('cpu')
modelo_cpu, ck_cpu = carregar_checkpoint('results/modelo.pt', cpu)
te_cpu = pd.read_csv(f"results/predicoes_teste_{ck_cpu['arquitetura']}.csv")
lat_cpu = medir_latencia(modelo_cpu, te_cpu, ck_cpu['tamanho'], cpu, n=100)
print(lat_cpu)
json.dump(lat_cpu, open('results/latencia_cpu.json', 'w'), indent=2)


In [ ]:
# Matriz de confusão do modelo escolhido, no teste
import numpy as np
r = met['modelos'][met['modelo_escolhido']]['teste_limiar_escolhido']
m = np.array([[r['tn'], r['fp']], [r['fn'], r['tp']]])
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(m, cmap='Blues')
for (i, j), v in np.ndenumerate(m):
    ax.text(j, i, v, ha='center', va='center', fontsize=14)
ax.set_xticks([0, 1]); ax.set_xticklabels(['sem fissura', 'com fissura'])
ax.set_yticks([0, 1]); ax.set_yticklabels(['sem fissura', 'com fissura'])
ax.set_xlabel('Previsto'); ax.set_ylabel('Real'); ax.set_title('Matriz de confusão (teste)')
plt.tight_layout(); plt.savefig('results/matriz_confusao.png', dpi=150); plt.show()


## 5. Olhar os erros
Abra as imagens salvas e descreva no relatório o que aparece nelas (sombra, textura, junta, mancha, fissura muito fina e assim por diante). O nome do arquivo traz a probabilidade que o modelo deu.

In [ ]:
import glob
nome = met['modelo_escolhido']
for tipo in ['falsos_positivos', 'falsos_negativos']:
    arqs = sorted(glob.glob(f'results/erros/{nome}/{tipo}/*.jpg'))[:12]
    if not arqs:
        print('Sem', tipo); continue
    fig, eixos = plt.subplots(2, 6, figsize=(16, 6))
    for ax in eixos.ravel(): ax.axis('off')
    for ax, a in zip(eixos.ravel(), arqs):
        ax.imshow(Image.open(a)); ax.set_title(pathlib.Path(a).name[:14], fontsize=8)
    fig.suptitle(tipo.replace('_', ' ')); plt.tight_layout(); plt.show()


## 6. Funcionalidade de monitoramento: imagem, sequência e vídeo
Duas opções:
* **A. Seu próprio vídeo curto** de pavimento ou parede (mp4). Suba o arquivo no Colab e coloque o caminho em `ENTRADA`.
* **B. Sequência simulada** com recortes do conjunto de teste. Como eu sei a resposta de cada quadro, dá para medir se o sistema acertou. É uma simulação e deve ser descrita assim no relatório.

Rode B primeiro, que não depende de nenhum arquivo. Se gravar também um vídeo real, mostre os dois no vídeo da entrega.

In [ ]:
import torch, pandas as pd
from video import carregar_checkpoint, processar_video, analisar_imagem
from sequencia import montar_sequencia, comparar_com_verdade

disp = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
modelo, ck = carregar_checkpoint('results/modelo.pt', disp)
LIMIAR, TAMANHO = ck['limiar'], ck['tamanho']

# Opção B: sequência simulada com recortes do teste
pred = pd.read_csv(f"results/predicoes_teste_{ck['arquitetura']}.csv")
verdade = montar_sequencia(pred, 'results/sequencia_simulada', quadros=30, inicio_evento=10, fim_evento=19)
res = processar_video(modelo, 'results/sequencia_simulada', 'results/video_saida.mp4', None,
                      LIMIAR, TAMANHO, disp, cada_n=1)
res['tipo_entrada'] = 'sequência simulada de 30 quadros montada com recortes do conjunto de teste'
res['comparacao_com_verdade'] = comparar_com_verdade(res, verdade)
json.dump(res, open('results/video_resumo.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
print({k: v for k, v in res.items() if k != 'quadros'})


In [ ]:
# Opção A: vídeo real (descomente e ajuste o caminho)
# ENTRADA = '/content/meu_video.mp4'
# res = processar_video(modelo, ENTRADA, 'results/video_real_saida.mp4', 'results/video_real.json',
#                       LIMIAR, TAMANHO, disp, cada_n=5, largura_alvo=1024, max_quadros=120)
# res['tipo_entrada'] = 'vídeo real gravado por mim em [descrever o local]'
# json.dump(res, open('results/video_resumo.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
# print({k: v for k, v in res.items() if k != 'quadros'})


In [ ]:
# Uma imagem com a marcação e a confiança
img, r = analisar_imagem(modelo, 'results/sequencia_simulada/quadro_012.jpg', LIMIAR, TAMANHO, disp)
import cv2
plt.figure(figsize=(12, 6)); plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); plt.axis('off'); plt.show()
print(r)


## 7. Gerar o relatório com os números medidos
Todos os números da tabelas vêm de `results/`. Depois, abra `docs/relatorio_visao.md` e preencha a parte da análise de erros com o que você viu nas imagens.

In [ ]:
!python src/gerar_relatorio.py results docs/relatorio_visao.md
print(open('docs/relatorio_visao.md', encoding='utf-8').read()[:3000])


## 8. Guardar os resultados no Drive
Copia modelo, métricas, erros, vídeo e relatório de volta para o Drive, porque os arquivos do Colab somem quando a sessão fecha.

In [ ]:
!mkdir -p /content/drive/MyDrive/via_cidadao_visao_resultados
!cp -r results docs /content/drive/MyDrive/via_cidadao_visao_resultados/
!ls /content/drive/MyDrive/via_cidadao_visao_resultados
